# MMA Fight Phase Classifier — CNN vs LSTM

This notebook trains two video classifiers on ~5-second MMA clips and compares them
using K-fold cross-validation.

**Before running:**
1. Go to **Runtime → Change runtime type → T4 GPU**
2. Upload your fight folders to Google Drive under `My Drive/mma_data/`

Expected Drive layout:
```
My Drive/
  mma_data/
    Rafael_Fiziev_vs_Ignacio_Bahamondes/
      Rafael_Fiziev_vs_Ignacio_Bahamondes_labels.csv
      clip_0002.mp4
      clip_0003.mp4
      ...
    Another_Fight/
      Another_Fight_labels.csv
      ...
```

## 1 — Setup

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q opencv-python-headless

In [ ]:
import cv2
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from torchvision import transforms
from collections import Counter

print(f"PyTorch:  {torch.__version__}")
print(f"CUDA:     {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:      {torch.cuda.get_device_name(0)}")

## 2 — Configuration

Edit `DATA_DIR` if your data is somewhere else on Drive.

In [ ]:
# ════════════════════════════════════════════════
#  CONFIGURATION — edit this cell to tune
# ════════════════════════════════════════════════

class cfg:
    # ── Paths ─────────────────────────────────────
    # Point this at the Drive folder that contains one sub-folder per fight
    DATA_DIR   = Path("/content/drive/MyDrive/mma_data")
    OUTPUT_DIR = Path("/content/outputs")

    # ── Labels ────────────────────────────────────
    PHASE_LABELS = [
        "Striking",
        "Grappling/Ground Work",
        "Clinch",
        "Transition/Takedown",
        "Neutral/Measuring Distance",
    ]
    NUM_CLASSES = len(PHASE_LABELS)
    LABEL2IDX = {label: idx for idx, label in enumerate(PHASE_LABELS)}
    IDX2LABEL = {idx: label for label, idx in LABEL2IDX.items()}

    # ── Video pre-processing ─────────────────────
    NUM_FRAMES    = 16
    FRAME_HEIGHT  = 112
    FRAME_WIDTH   = 112

    # ── Training ─────────────────────────────────
    BATCH_SIZE      = 8
    NUM_EPOCHS      = 30
    LEARNING_RATE   = 1e-4
    WEIGHT_DECAY    = 1e-4
    PATIENCE        = 7
    RANDOM_SEED     = 42
    NUM_WORKERS     = 2
    K_FOLDS         = 5

    # ── Augmentation ─────────────────────────────
    TEMPORAL_JITTER = 0.15
    MIXUP_ALPHA     = 0.2       # set 0.0 to disable

    # ── Model-specific ───────────────────────────
    LSTM_HIDDEN      = 256
    LSTM_LAYERS      = 2
    LSTM_DROPOUT     = 0.3
    CNN_BASE_FILTERS = 32

    # ── Device ───────────────────────────────────
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


cfg.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
torch.manual_seed(cfg.RANDOM_SEED)
print(f"Device: {cfg.DEVICE}")
print(f"Data:   {cfg.DATA_DIR}")
print(f"Exists: {cfg.DATA_DIR.exists()}")

## 3 — Dataset

In [ ]:
# ════════════════════════════════════════════════
#  TRANSFORMS
# ════════════════════════════════════════════════

train_spatial_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomResizedCrop((cfg.FRAME_HEIGHT, cfg.FRAME_WIDTH), scale=(0.75, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05),
    transforms.RandomGrayscale(p=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_spatial_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((cfg.FRAME_HEIGHT, cfg.FRAME_WIDTH)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])


# ════════════════════════════════════════════════
#  DATA DISCOVERY
# ════════════════════════════════════════════════

def discover_clips(data_dir: Path) -> pd.DataFrame:
    """Scan fight sub-folders, return merged DataFrame of valid clips."""
    all_rows = []
    data_dir = Path(data_dir)

    for fight_dir in sorted(data_dir.iterdir()):
        if not fight_dir.is_dir():
            continue
        csvs = list(fight_dir.glob("*_labels.csv"))
        if not csvs:
            print(f"  ⚠  No *_labels.csv in {fight_dir.name}, skipping")
            continue
        csv_path = csvs[0]
        df = pd.read_csv(csv_path)

        if df["excluded"].dtype == object:
            df["excluded"] = df["excluded"].str.strip().str.lower() == "true"

        df = df[~df["excluded"]].dropna(subset=["phase_label"]).copy()
        df["clip_path"] = df["saved_filename"].apply(lambda f: str(fight_dir / f))
        df["fight"] = fight_dir.name
        all_rows.append(df)

    if not all_rows:
        raise FileNotFoundError(
            f"No valid fight folders found under {data_dir}. "
            "Each sub-folder needs a *_labels.csv and clip mp4s."
        )

    merged = pd.concat(all_rows, ignore_index=True)
    print(f"Discovered {len(merged)} valid clips across {len(all_rows)} fight(s)")
    return merged


# ════════════════════════════════════════════════
#  VIDEO DATASET
# ════════════════════════════════════════════════

class VideoDataset(Dataset):
    def __init__(self, records: pd.DataFrame, transform=None,
                 temporal_jitter: float = 0.0):
        self.records = records.reset_index(drop=True)
        self.transform = transform or val_spatial_transform
        self.temporal_jitter = temporal_jitter

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        row = self.records.iloc[idx]
        video_path = row["clip_path"]
        label = cfg.LABEL2IDX[row["phase_label"]]

        frames = self._load_frames(video_path)
        frames = self._sample_frames(frames)
        tensor = self._to_tensor(frames)
        return tensor, label

    def _load_frames(self, path: str):
        cap = cv2.VideoCapture(path)
        frames = []
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        cap.release()
        if len(frames) == 0:
            raise RuntimeError(f"Could not read any frames from {path}")
        return frames

    def _sample_frames(self, frames):
        n = len(frames)
        if n < cfg.NUM_FRAMES:
            indices = list(range(n)) + [n - 1] * (cfg.NUM_FRAMES - n)
            return [frames[i] for i in indices]

        ideal = np.linspace(0, n - 1, cfg.NUM_FRAMES)
        if self.temporal_jitter > 0:
            gap = (n - 1) / max(cfg.NUM_FRAMES - 1, 1)
            max_shift = self.temporal_jitter * gap
            jitter = np.random.uniform(-max_shift, max_shift, size=cfg.NUM_FRAMES)
            ideal = ideal + jitter

        indices = np.clip(ideal.round().astype(int), 0, n - 1)
        return [frames[i] for i in indices]

    def _to_tensor(self, frames):
        tensors = [self.transform(f) for f in frames]
        video = torch.stack(tensors, dim=1)
        return video


# ════════════════════════════════════════════════
#  MIXUP
# ════════════════════════════════════════════════

def mixup_batch(videos, labels, alpha=cfg.MIXUP_ALPHA):
    if alpha <= 0:
        return videos, (labels, labels, 1.0)
    lam = np.random.beta(alpha, alpha)
    lam = max(lam, 1 - lam)
    perm = torch.randperm(videos.size(0))
    mixed = lam * videos + (1 - lam) * videos[perm]
    return mixed, (labels, labels[perm], lam)


# ════════════════════════════════════════════════
#  CLASS WEIGHTS
# ════════════════════════════════════════════════

def compute_class_weights(labels):
    counts = Counter(labels)
    n = len(labels)
    weights = torch.zeros(cfg.NUM_CLASSES)
    for cls_idx in range(cfg.NUM_CLASSES):
        c = counts.get(cls_idx, 1)
        weights[cls_idx] = n / (cfg.NUM_CLASSES * c)
    return weights


print("Dataset utilities ready ✓")

### Verify data loads correctly
Run this cell to check your Drive data is accessible and see the class distribution.

In [ ]:
# Quick data sanity check
records = discover_clips(cfg.DATA_DIR)
labels = records["phase_label"].map(cfg.LABEL2IDX).values

print("\nClass distribution:")
for lbl, idx in cfg.LABEL2IDX.items():
    count = (labels == idx).sum()
    print(f"  [{idx}] {lbl:35s} — {count:>4d} clips")

# Load one clip to verify video reading works
test_ds = VideoDataset(records.head(1), transform=val_spatial_transform)
video_tensor, label = test_ds[0]
print(f"\nSample tensor shape: {video_tensor.shape}")
print(f"Label: {cfg.IDX2LABEL[label]}")
print("\n✓ Data is ready")

## 4 — Models

In [ ]:
# ════════════════════════════════════════════════
#  MODEL 1 — Conv3D CNN
# ════════════════════════════════════════════════

class Conv3DClassifier(nn.Module):
    """
    4 × [Conv3d → BN → ReLU → MaxPool3d] → AdaptiveAvgPool → FC
    Learns spatiotemporal features jointly via 3D convolutions.
    """
    def __init__(self, num_classes=cfg.NUM_CLASSES, base_filters=cfg.CNN_BASE_FILTERS):
        super().__init__()

        def _block(in_c, out_c, pool_kernel=(2, 2, 2)):
            return nn.Sequential(
                nn.Conv3d(in_c, out_c, kernel_size=3, padding=1),
                nn.BatchNorm3d(out_c),
                nn.ReLU(inplace=True),
                nn.MaxPool3d(kernel_size=pool_kernel),
            )

        f = base_filters
        self.features = nn.Sequential(
            _block(3,     f,     pool_kernel=(1, 2, 2)),
            _block(f,     f * 2, pool_kernel=(2, 2, 2)),
            _block(f * 2, f * 4, pool_kernel=(2, 2, 2)),
            _block(f * 4, f * 8, pool_kernel=(2, 2, 2)),
        )
        self.pool = nn.AdaptiveAvgPool3d(1)
        self.head = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(f * 8, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x).flatten(1)
        return self.head(x)


# ════════════════════════════════════════════════
#  Lightweight 2D frame encoder
# ════════════════════════════════════════════════

class _FrameEncoder(nn.Module):
    """4-block 2D CNN: (3, 112, 112) → (out_dim,)"""
    def __init__(self, out_dim=256):
        super().__init__()

        def _block(in_c, out_c):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_c),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2),
            )

        self.features = nn.Sequential(
            _block(3,   64),
            _block(64,  128),
            _block(128, 256),
            _block(256, out_dim),
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.out_dim = out_dim

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x).flatten(1)
        return x


# ════════════════════════════════════════════════
#  MODEL 2 — Frame-encoder → LSTM
# ════════════════════════════════════════════════

class LSTMClassifier(nn.Module):
    """
    Encodes each frame independently (2D CNN → 256-d), then feeds the
    sequence of frame features into a 2-layer LSTM.
    """
    def __init__(self, num_classes=cfg.NUM_CLASSES, encoder_dim=256,
                 hidden_size=cfg.LSTM_HIDDEN, num_layers=cfg.LSTM_LAYERS,
                 dropout=cfg.LSTM_DROPOUT):
        super().__init__()
        self.encoder = _FrameEncoder(out_dim=encoder_dim)
        self.lstm = nn.LSTM(
            input_size=encoder_dim, hidden_size=hidden_size,
            num_layers=num_layers, batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, num_classes),
        )

    def forward(self, x):
        B, C, T, H, W = x.shape
        x = x.permute(0, 2, 1, 3, 4).contiguous().view(B * T, C, H, W)
        feats = self.encoder(x).view(B, T, -1)
        lstm_out, _ = self.lstm(feats)
        last_hidden = lstm_out[:, -1, :]
        return self.head(last_hidden)


# ── Factory ──
def build_model(name: str) -> nn.Module:
    if name == "cnn":
        model = Conv3DClassifier()
    elif name == "lstm":
        model = LSTMClassifier()
    else:
        raise ValueError(f"Unknown model: {name!r}")
    return model.to(cfg.DEVICE)


# Quick shape check
dummy = torch.randn(2, 3, cfg.NUM_FRAMES, cfg.FRAME_HEIGHT, cfg.FRAME_WIDTH).to(cfg.DEVICE)
for name in ["cnn", "lstm"]:
    m = build_model(name)
    p = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f"{name.upper():5s}  output={m(dummy).shape}  trainable_params={p:,}")
del dummy
print("\nModels ready ✓")

## 5 — Training Loop

In [ ]:
# ════════════════════════════════════════════════
#  TRAINING (with early stopping, weighted loss, mixup)
# ════════════════════════════════════════════════

def train_model(model, train_loader, val_loader, model_name,
                class_weights=None, fold=None):
    tag = f"{model_name}_fold{fold}" if fold is not None else model_name
    ckpt_path = cfg.OUTPUT_DIR / f"best_{tag}.pt"

    if class_weights is not None:
        class_weights = class_weights.to(cfg.DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=cfg.LEARNING_RATE, weight_decay=cfg.WEIGHT_DECAY,
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=3, verbose=False,
    )

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_val_loss = float("inf")
    patience_counter = 0
    use_mixup = cfg.MIXUP_ALPHA > 0

    for epoch in range(1, cfg.NUM_EPOCHS + 1):
        t0 = time.time()
        model.train()
        running_loss, correct, total = 0.0, 0, 0

        for videos, labels in train_loader:
            videos, labels = videos.to(cfg.DEVICE), labels.to(cfg.DEVICE)

            if use_mixup:
                videos, (labels_a, labels_b, lam) = mixup_batch(videos, labels)
                videos = videos.to(cfg.DEVICE)
                labels_a, labels_b = labels_a.to(cfg.DEVICE), labels_b.to(cfg.DEVICE)

            optimizer.zero_grad()
            logits = model(videos)

            if use_mixup:
                loss = lam * criterion(logits, labels_a) + (1 - lam) * criterion(logits, labels_b)
                correct += (logits.argmax(1) == labels_a).sum().item()
            else:
                loss = criterion(logits, labels)
                correct += (logits.argmax(1) == labels).sum().item()

            loss.backward()
            optimizer.step()

            running_loss += loss.item() * videos.size(0)
            total += videos.size(0)

        train_loss = running_loss / total
        train_acc  = correct / total

        val_loss, val_acc = evaluate_loss(model, val_loader, criterion)
        scheduler.step(val_loss)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        elapsed = time.time() - t0
        print(f"  [{tag}] Epoch {epoch:>2}/{cfg.NUM_EPOCHS}  "
              f"t_loss={train_loss:.4f}  t_acc={train_acc:.3f}  "
              f"v_loss={val_loss:.4f}  v_acc={val_acc:.3f}  "
              f"({elapsed:.1f}s)")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save(model.state_dict(), ckpt_path)
        else:
            patience_counter += 1
            if patience_counter >= cfg.PATIENCE:
                print(f"  Early stopping at epoch {epoch}")
                break

    model.load_state_dict(torch.load(ckpt_path, map_location=cfg.DEVICE,
                                     weights_only=True))
    return history


@torch.no_grad()
def evaluate_loss(model, loader, criterion):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    for videos, labels in loader:
        videos, labels = videos.to(cfg.DEVICE), labels.to(cfg.DEVICE)
        logits = model(videos)
        loss = criterion(logits, labels)
        running_loss += loss.item() * videos.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        total += videos.size(0)
    return running_loss / total, correct / total


@torch.no_grad()
def collect_predictions(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    for videos, labels in loader:
        videos = videos.to(cfg.DEVICE)
        logits = model(videos)
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())
    return np.array(all_labels), np.array(all_preds)


print("Training utilities ready ✓")

## 6 — K-Fold Cross-Validation Runner

In [ ]:
# ════════════════════════════════════════════════
#  K-FOLD CV RUNNER
# ════════════════════════════════════════════════

def build_kfold_loaders(data_dir=cfg.DATA_DIR):
    """Yields (fold_idx, train_loader, val_loader, class_weights) per fold."""
    records = discover_clips(data_dir)
    labels = records["phase_label"].map(cfg.LABEL2IDX).values

    min_class_count = min(Counter(labels).values())
    can_stratify = min_class_count >= cfg.K_FOLDS

    if can_stratify:
        kf = StratifiedKFold(n_splits=cfg.K_FOLDS, shuffle=True,
                             random_state=cfg.RANDOM_SEED)
        split_iter = kf.split(records, labels)
        print(f"Using {cfg.K_FOLDS}-fold Stratified CV")
    else:
        kf = KFold(n_splits=cfg.K_FOLDS, shuffle=True,
                   random_state=cfg.RANDOM_SEED)
        split_iter = kf.split(records)
        print(f"Using {cfg.K_FOLDS}-fold CV (non-stratified, "
              f"smallest class = {min_class_count})")

    for fold_idx, (train_idx, val_idx) in enumerate(split_iter):
        train_ds = VideoDataset(records.iloc[train_idx],
                                transform=train_spatial_transform,
                                temporal_jitter=cfg.TEMPORAL_JITTER)
        val_ds   = VideoDataset(records.iloc[val_idx],
                                transform=val_spatial_transform,
                                temporal_jitter=0.0)

        train_loader = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE,
                                  shuffle=True, num_workers=cfg.NUM_WORKERS,
                                  pin_memory=True)
        val_loader   = DataLoader(val_ds, batch_size=cfg.BATCH_SIZE,
                                  shuffle=False, num_workers=cfg.NUM_WORKERS,
                                  pin_memory=True)

        class_weights = compute_class_weights(labels[train_idx])
        print(f"  Fold {fold_idx + 1}/{cfg.K_FOLDS}  "
              f"train={len(train_idx)}  val={len(val_idx)}")
        yield fold_idx, train_loader, val_loader, class_weights


def run_kfold(model_name):
    """Train model_name across K folds, return aggregated metrics."""
    all_y_true, all_y_pred = [], []
    fold_val_accs = []

    for fold_idx, train_loader, val_loader, class_weights in build_kfold_loaders():
        print(f"\n{'─'*50}")
        print(f"  {model_name.upper()} — Fold {fold_idx + 1}/{cfg.K_FOLDS}")
        print(f"{'─'*50}")

        model = build_model(model_name)
        if fold_idx == 0:
            n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
            print(f"  Trainable parameters: {n_params:,}")

        history = train_model(model, train_loader, val_loader,
                              model_name=model_name,
                              class_weights=class_weights,
                              fold=fold_idx)

        y_true, y_pred = collect_predictions(model, val_loader)
        all_y_true.append(y_true)
        all_y_pred.append(y_pred)

        best_val_acc = max(history["val_acc"])
        fold_val_accs.append(best_val_acc)
        print(f"  Fold {fold_idx + 1} best val acc: {best_val_acc:.3f}")

    all_y_true = np.concatenate(all_y_true)
    all_y_pred = np.concatenate(all_y_pred)
    return all_y_true, all_y_pred, fold_val_accs


print("K-Fold runner ready ✓")

## 7 — Train CNN

In [ ]:
cnn_y_true, cnn_y_pred, cnn_fold_accs = run_kfold("cnn")

## 8 — Train LSTM

In [ ]:
lstm_y_true, lstm_y_pred, lstm_fold_accs = run_kfold("lstm")

## 9 — Results & Comparison

In [ ]:
# ════════════════════════════════════════════════
#  PER-MODEL REPORTS
# ════════════════════════════════════════════════

def show_results(name, y_true, y_pred, fold_accs):
    mean_acc = np.mean(fold_accs)
    std_acc  = np.std(fold_accs)
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)

    print(f"\n{'═'*60}")
    print(f"  {name.upper()} — {cfg.K_FOLDS}-Fold Results")
    print(f"{'═'*60}")
    print(f"  Per-fold val accs: {[f'{a:.3f}' for a in fold_accs]}")
    print(f"  Mean ± Std:  {mean_acc:.3f} ± {std_acc:.3f}")
    print(f"  Macro F1:    {macro_f1:.3f}\n")
    print(classification_report(
        y_true, y_pred,
        target_names=cfg.PHASE_LABELS,
        zero_division=0,
    ))

    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=cfg.PHASE_LABELS,
                yticklabels=cfg.PHASE_LABELS, ax=ax)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(f"Confusion Matrix — {name.upper()} (all folds)")
    plt.xticks(rotation=30, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(cfg.OUTPUT_DIR / f"confusion_{name}.png", dpi=150)
    plt.show()

    return {"mean_acc": mean_acc, "std_acc": std_acc, "macro_f1": macro_f1}


cnn_summary  = show_results("cnn",  cnn_y_true,  cnn_y_pred,  cnn_fold_accs)
lstm_summary = show_results("lstm", lstm_y_true, lstm_y_pred, lstm_fold_accs)

In [ ]:
# ════════════════════════════════════════════════
#  SIDE-BY-SIDE COMPARISON
# ════════════════════════════════════════════════

summaries = {"CNN": cnn_summary, "LSTM": lstm_summary}
names = list(summaries.keys())
means = [summaries[n]["mean_acc"] for n in names]
stds  = [summaries[n]["std_acc"]  for n in names]
f1s   = [summaries[n]["macro_f1"] for n in names]

x = np.arange(len(names))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x - width/2, means, width, yerr=stds, label="Val Accuracy",
       capsize=5, color="#4C72B0")
ax.bar(x + width/2, f1s,   width, label="Macro F1",
       color="#DD8452")
ax.set_xticks(x)
ax.set_xticklabels(names)
ax.set_ylim(0, 1)
ax.set_ylabel("Score")
ax.set_title(f"{cfg.K_FOLDS}-Fold CV — Model Comparison")
ax.legend()
plt.tight_layout()
plt.savefig(cfg.OUTPUT_DIR / "model_comparison.png", dpi=150)
plt.show()

# Final summary table
print(f"\n{'═'*50}")
print(f"  FINAL COMPARISON")
print(f"{'═'*50}")
for name, s in summaries.items():
    print(f"  {name:5s}  acc={s['mean_acc']:.3f}±{s['std_acc']:.3f}  "
          f"macro_f1={s['macro_f1']:.3f}")
print(f"\n✓ All outputs saved to {cfg.OUTPUT_DIR}/")

## 10 — Save outputs to Drive (optional)

Copy checkpoints and plots back to your Google Drive so they persist after the runtime disconnects.

In [ ]:
import shutil

drive_output = Path("/content/drive/MyDrive/mma_outputs")
drive_output.mkdir(parents=True, exist_ok=True)

for f in cfg.OUTPUT_DIR.iterdir():
    shutil.copy2(f, drive_output / f.name)
    print(f"  Copied {f.name}")

print(f"\n✓ All outputs saved to Drive: {drive_output}")